# Grafo bipartito

Un grafo es bipartito si sus nodos se pueden repartir en dos grupos de forma que toda arista une nodos de grupos distintos. Equivale a poder colorearlo con dos colores y a no tener ciclos de longitud impar. Se comprueba con un BFS que asigna colores alternos por niveles en $O(V + E)$; si una arista une dos nodos del mismo color, hay un ciclo impar.

## 1. Comprobar con dos colores

Escribir una función que devuelva el reparto en dos grupos si el grafo es bipartito, o `None` si no lo es. Probarla con un grafo de alumnos y asignaturas y con un triángulo.

In [1]:
import random
from collections import deque
from itertools import product

def bipartito(grafo):
    color = {}
    for inicio in grafo:
        if inicio in color:
            continue
        color[inicio] = 0
        cola = deque([inicio])
        while cola:
            u = cola.popleft()
            for v in grafo[u]:
                if v not in color:
                    color[v] = 1 - color[u]
                    cola.append(v)
                elif color[v] == color[u]:
                    return None
    return ({u for u in color if color[u] == 0}, {u for u in color if color[u] == 1})

def no_dirigido(aristas, nodos=()):
    g = {u: set() for u in nodos}
    for u, v in aristas:
        g.setdefault(u, set()).add(v)
        g.setdefault(v, set()).add(u)
    return g

matriculas = no_dirigido([("Ana", "Mates"), ("Ana", "Física"), ("Luis", "Mates"), ("Eva", "Historia"), ("Eva", "Física")])
print(bipartito(matriculas))
print(bipartito(no_dirigido([(1, 2), (2, 3), (3, 1)])))

({'Luis', 'Ana', 'Eva'}, {'Mates', 'Historia', 'Física'})
None


## 2. Contra fuerza bruta

Comprobar `bipartito` en grafos aleatorios pequeños contra probar los $2^V$ coloreados posibles.

In [2]:
def bipartito_bruto(n, aristas):
    return any(all(c[u] != c[v] for u, v in aristas) for c in product((0, 1), repeat=n))

random.seed(1)
for _ in range(500):
    n = random.randint(1, 8)
    aristas = [(u, v) for u in range(n) for v in range(u + 1, n) if random.random() < 0.3]
    reparto = bipartito(no_dirigido(aristas, range(n)))
    assert (reparto is not None) == bipartito_bruto(n, aristas)
    if reparto:
        a, b = reparto
        assert all((u in a) != (v in a) for u, v in aristas)
print("ok")

ok


## 3. Encontrar un ciclo impar

Cuando el grafo no es bipartito, devolver un ciclo de longitud impar como prueba: al encontrar la arista conflictiva $(u, v)$, subir desde ambos por el árbol del BFS hasta su antecesor común.

In [3]:
def ciclo_impar(grafo):
    for inicio in grafo:
        padre, nivel = {inicio: None}, {inicio: 0}
        cola = deque([inicio])
        while cola:
            u = cola.popleft()
            for v in grafo[u]:
                if v not in nivel:
                    padre[v], nivel[v] = u, nivel[u] + 1
                    cola.append(v)
                elif nivel[v] == nivel[u]:
                    a, b = [u], [v]
                    while a[-1] != b[-1]:
                        a.append(padre[a[-1]])
                        b.append(padre[b[-1]])
                    return a + b[-2::-1]
    return None

g = no_dirigido([(0, 1), (1, 2), (2, 3), (3, 4), (4, 0), (0, 5)])
ciclo = ciclo_impar(g)
print(ciclo)
assert len(ciclo) % 2 == 1 and all(ciclo[i + 1] in g[ciclo[i]] for i in range(len(ciclo) - 1)) and ciclo[0] in g[ciclo[-1]]

for _ in range(300):
    n = random.randint(3, 9)
    aristas = [(u, v) for u in range(n) for v in range(u + 1, n) if random.random() < 0.35]
    g = no_dirigido(aristas, range(n))
    c = ciclo_impar(g)
    assert (c is None) == (bipartito(g) is not None)
    if c:
        assert len(c) % 2 == 1 and len(set(c)) == len(c)

[2, 1, 0, 4, 3]


## 4. Emparejamiento máximo

En un grafo bipartito de voluntarios y turnos (cada voluntario puede cubrir algunos turnos), emparejar el máximo número de voluntarios con turnos distintos usando caminos aumentantes (algoritmo de Kuhn, $O(V \cdot E)$). Comprobarlo contra fuerza bruta.

In [4]:
def emparejamiento(izquierda, aristas):
    pareja = {}  # turno -> voluntario

    def aumenta(u, vistos):
        for v in aristas.get(u, ()):
            if v not in vistos:
                vistos.add(v)
                if v not in pareja or aumenta(pareja[v], vistos):
                    pareja[v] = u
                    return True
        return False

    for u in izquierda:
        aumenta(u, set())
    return {u: v for v, u in pareja.items()}

disponibilidad = {"Ana": ["lunes", "martes"], "Luis": ["lunes"], "Eva": ["martes", "jueves"], "Pau": ["lunes"], "Noa": ["jueves", "viernes"]}
asignacion = emparejamiento(disponibilidad, disponibilidad)
print(asignacion)

def emparejamiento_bruto(izq, aristas):
    pares = [(u, v) for u in izq for v in aristas.get(u, ())]
    mejor = 0
    for mascara in range(1 << len(pares)):
        elegidos = [pares[i] for i in range(len(pares)) if mascara >> i & 1]
        if len({u for u, _ in elegidos}) == len({v for _, v in elegidos}) == len(elegidos):
            mejor = max(mejor, len(elegidos))
    return mejor

for _ in range(200):
    izq = range(random.randint(1, 5))
    aristas = {u: [v for v in "abcd" if random.random() < 0.4] for u in izq}
    m = emparejamiento(izq, aristas)
    assert len(set(m.values())) == len(m) and all(v in aristas[u] for u, v in m.items())
    assert len(m) == emparejamiento_bruto(izq, aristas)

{'Luis': 'lunes', 'Ana': 'martes', 'Eva': 'jueves', 'Noa': 'viernes'}
